# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task Type:** Classification and Ranking/Scoring.

**Why:** We are predicting a binary outcome (Will this page decline in the future?), which makes it a Classification task. However, because content teams have limited resources, they cannot refresh every single declining page. Therefore, the model must output a probability score so we can *rank* the pages from highest risk to lowest risk.

## 2. Target or proxy

**Target:** A binary label (`1` = Will Decline, `0` = Stable/Growth).

**Where does it come from:** This must be an *observed outcome*, not a defined rule. We cannot use the `is_declining_label` from the starter dataset because it is a hand-written rule based on the past 90 days. Instead, our true target will be observed directly from the time-series warehouse data: we will measure if a page's actual impressions dropped significantly in a future 30-day window compared to its prior window.

## 3. Success metric

**Success Metric:** Precision@K and ROC-AUC.

**What means 'good':** Overall accuracy is a poor metric here because the content team can only realistically update a limited number of pages per week (e.g., the top 50). **Precision@K** tells us exactly what matters: *'Out of the top 50 pages the model flagged as highest risk, how many actually went on to decline?'* We will also use **ROC-AUC** to measure the model's overall ability to separate declining pages from stable ones.

## 4. The unit of analysis, as a real dataframe

We load the starter dataset below to prove our unit of analysis. **One row = one pseudonymized content item (page)**. 

We also sketch out what the true target column (`is_declining_next_30`) would look like once we calculate it from the future time-series data.

In [1]:
import pandas as pd
import numpy as np

# Load the starter slice
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Sketching the future target column (for illustration purposes on this snapshot)
# In reality, this will be calculated using forward-window SQL aggregations from the warehouse.
df['is_declining_next_30_mock'] = np.where(df['trend_pct'] <= -0.20, 1, 0) 

# Show the unit of analysis: one row = one content item
df[['client_id', 'content_id', 'content_age_days', 'impressions_90d', 'is_declining_next_30_mock']].head()

,client_id,content_id,content_age_days,impressions_90d,is_declining_next_30_mock
0,client_f369cb89fc,content_304f48230142,187,3803,1
1,client_4e07408562,content_a1fb4e703a9e,445,15320,1
2,client_7f2253d7e2,content_9aa793d4d895,141,12581,1
3,client_19581e27de,content_331d6c4de07b,463,11751,1
4,client_3fdba35f04,content_d99b7a2d90ca,263,19140,1


## 5. Why ML beats a fixed rule here

A fixed rule (e.g., *'flag any page over 180 days old with declining CTR'*) is too rigid. It will flag perfectly healthy evergreen pages while completely missing young, volatile pages that are crashing early. 

ML earns its place here because the patterns of content decay are messy, tangled, and shift over time. Position volatility, query concentration, click-through rates, and age all interact simultaneously in non-linear ways that a human simply cannot write a comprehensive if-statement for.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.